# Synthetic Generation (Simple Colab Mode)

This notebook is simplified to one path only:
1. Load local model in this Colab runtime (`transformers` + 4-bit).
2. Read parsed terms from Drive (`/content/drive/MyDrive/dap-hs/singlish_parsed.jsonl`).
3. Generate synthetic outputs directly to Drive (`synthetic_training.jsonl`).

No ngrok, no external backend, no repo imports.

In [1]:
# Optional: install deps if needed
%pip install -q requests python-dotenv

## Load local model (same Colab runtime)
This will download/load the model into Colab GPU memory.

In [2]:
# One-cell local model setup (Qwen2.5 text 4-bit)
import sys
import torch
import subprocess

print("[1/3] Installing local inference dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-Uq", "transformers", "accelerate", "bitsandbytes"],
    check=True,
 )
print("✅ Dependencies installed")

print("[2/3] Loading model/tokenizer...")
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

LOCAL_MODEL = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    ),
)
LOCAL_MODEL.eval()

LOCAL_TOKENIZER = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if LOCAL_TOKENIZER.pad_token is None:
    LOCAL_TOKENIZER.pad_token = LOCAL_TOKENIZER.eos_token

HF_MODEL = model_name
print("[3/3] Local model ready")
print("✅ HF model:", HF_MODEL)
print("✅ Device:", next(LOCAL_MODEL.parameters()).device)

def local_chat_generate(system_prompt: str, user_prompt: str, temperature: float = 0.8, max_new_tokens: int = 768) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    chat_text = LOCAL_TOKENIZER.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = LOCAL_TOKENIZER(chat_text, return_tensors="pt").to(LOCAL_MODEL.device)

    with torch.no_grad():
        outputs = LOCAL_MODEL.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            pad_token_id=LOCAL_TOKENIZER.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return LOCAL_TOKENIZER.decode(generated, skip_special_tokens=True).strip()

[1/3] Installing local inference dependencies...
✅ Dependencies installed
[2/3] Loading model/tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[3/3] Local model ready
✅ HF model: Qwen/Qwen2.5-7B-Instruct
✅ Device: cuda:0


In [3]:
# Mount Google Drive + set a dedicated working folder for this project
USE_GOOGLE_DRIVE = True

from pathlib import Path

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('Google Drive mounted at /content/drive')
    except Exception as e:
        print('Drive mount skipped or unavailable:', e)

drive_root = Path('/content/drive/MyDrive')
DAP_HS_DIR = drive_root / 'dap-hs'
DAP_HS_DIR.mkdir(parents=True, exist_ok=True)

# Optional repo path if you later copy the whole repo into Drive
DRIVE_REPO_PATH = DAP_HS_DIR / '11th-DAP-hate-speech'

# Parsed/output file paths used by fallback pipeline
DRIVE_PARSED_FILE = DAP_HS_DIR / 'singlish_parsed.jsonl'
DRIVE_OUTPUT_FILE = DAP_HS_DIR / 'synthetic_training.jsonl'

if DRIVE_REPO_PATH.exists():
    PROJECT_ROOT = DRIVE_REPO_PATH
    print('Using PROJECT_ROOT from Drive repo:', PROJECT_ROOT)
else:
    PROJECT_ROOT = Path.cwd()
    print('No Drive repo folder found; using runtime PROJECT_ROOT:', PROJECT_ROOT)

print('DAP_HS_DIR:', DAP_HS_DIR)
print('DRIVE_PARSED_FILE:', DRIVE_PARSED_FILE, '| exists:', DRIVE_PARSED_FILE.exists())
print('DRIVE_OUTPUT_FILE:', DRIVE_OUTPUT_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted at /content/drive
No Drive repo folder found; using runtime PROJECT_ROOT: /content
DAP_HS_DIR: /content/drive/MyDrive/dap-hs
DRIVE_PARSED_FILE: /content/drive/MyDrive/dap-hs/singlish_parsed.jsonl | exists: True
DRIVE_OUTPUT_FILE: /content/drive/MyDrive/dap-hs/synthetic_training.jsonl


In [4]:
# Upload local parsed file from your PC into Drive (run once if file is missing)
from pathlib import Path

if not DRIVE_PARSED_FILE.exists():
    print('DRIVE_PARSED_FILE not found, please upload singlish_parsed.jsonl from your computer...')
    from google.colab import files
    uploaded = files.upload()

    if uploaded:
        first_name = next(iter(uploaded.keys()))
        if first_name != 'singlish_parsed.jsonl':
            print(f'Uploaded {first_name}; renaming to singlish_parsed.jsonl')
        Path('singlish_parsed.jsonl').write_bytes(uploaded[first_name])
        DRIVE_PARSED_FILE.write_bytes(Path('singlish_parsed.jsonl').read_bytes())
        print('Saved to:', DRIVE_PARSED_FILE)
    else:
        print('No file uploaded.')
else:
    print('Found parsed file at:', DRIVE_PARSED_FILE)

Found parsed file at: /content/drive/MyDrive/dap-hs/singlish_parsed.jsonl


In [5]:
from pathlib import Path
import json
import re
import time

# Keep robust group descriptions (same style as scraping/synthetic_generation.py)
SG_TARGET_GROUPS = {
    'chinese': 'Chinese Singaporeans (majority race, includes dialect groups: Hokkien, Teochew, Cantonese, Hakka, Hainanese)',
    'malay': 'Malay Singaporeans (includes Muslim Malays)',
    'indian': 'Indian Singaporeans (includes Tamil, Hindi, Sikh, Malayalee communities)',
    'others': 'Eurasians, mixed-race, and other minority Singaporeans',
    'migrant_worker': 'Foreign migrant workers (Bangladeshi, Chinese nationals/PRCs, Vietnamese, Filipino, Myanmar, Indonesian)',
    'religion': 'Religious groups (Islam, Christianity, Buddhism, Hinduism, Taoism, Sikhism)',
}

LANG_TO_COMMUNITY = {
    'hokkien': 'chinese', 'teochew': 'chinese', 'cantonese': 'chinese',
    'hakka': 'chinese', 'hainanese': 'chinese', 'mandarin': 'chinese',
    'min nan': 'chinese', 'chinese': 'chinese',
    'malay': 'malay', 'bahasa': 'malay', 'jawi': 'malay',
    'tamil': 'indian', 'hindi': 'indian', 'punjabi': 'indian',
    'sanskrit': 'indian', 'urdu': 'indian',
    'bengali': 'migrant_worker', 'tagalog': 'migrant_worker',
    'filipino': 'migrant_worker', 'vietnamese': 'migrant_worker',
    'burmese': 'migrant_worker', 'myanmar': 'migrant_worker',
    'arabic': 'religion',
}

SYSTEM_PROMPT = """You generate Singapore social media posts for a hate speech detection research dataset.

You will be given a Singlish term and its meaning. Write posts that use that exact term.

Output ONLY a JSON array. No markdown, no explanation, no extra text.

Each object: {\"text\": \"...\", \"label\": \"hate|offensive|normal\", \"target_group\": \"...\"}

Labels:
- hate: targets a race/religion/nationality (chinese, malay, indian, migrant_worker, islam, etc.)
- offensive: rude but not targeting a protected group. target_group = \"none\"
- normal: neutral usage. target_group = \"none\"

Language rule (strict): Write in English only. You may use Singlish particles (lah, leh, lor, sia) and the given term, but do not write full Malay/Chinese/Tamil sentences.
"""

def infer_target_groups(languages):
    groups = {LANG_TO_COMMUNITY[l] for l in (languages or []) if l in LANG_TO_COMMUNITY}
    return list(groups) if groups else list(SG_TARGET_GROUPS.keys())

def extract_context_from_parsed(entry):
    definitions = (entry.get('definitions') or '').strip()
    usage_notes = (entry.get('usage_notes') or '').strip()
    synonyms = entry.get('synonyms') or []
    antonyms = entry.get('antonyms') or []

    parts = []
    if definitions:
        parts.append(f'Definitions:\n{definitions}')
    if usage_notes:
        parts.append(f'Usage notes:\n{usage_notes}')
    if synonyms:
        parts.append('Synonyms: ' + ', '.join(synonyms))
    if antonyms:
        parts.append('Antonyms: ' + ', '.join(antonyms))

    text = '\n\n'.join(parts).strip()
    return text[:1200] + ('\n...' if len(text) > 1200 else '')

def normalize_entry(entry):
    categories = entry.get('categories') or entry.get('source_categories') or []
    if isinstance(categories, str):
        categories = [categories]
    return {
        'term': (entry.get('term') or '').strip(),
        'categories': categories,
        'languages': entry.get('languages') or [],
        'context': extract_context_from_parsed(entry),
    }

def build_prompt(term, categories, context, n_per_label, target_groups, languages):
    cat_str = ', '.join(categories) if categories else 'general'
    lang_str = ', '.join(languages) if languages else 'mixed/unclear'

    group_descriptions = []
    for g in target_groups:
        if g in SG_TARGET_GROUPS:
            group_descriptions.append(f'- {g}: {SG_TARGET_GROUPS[g]}')
    group_block = '\n'.join(group_descriptions) if group_descriptions else '- any Singapore racial/religious/migrant group'

    return f'''Write {n_per_label * 3} Singapore social media posts that all use the Singlish term "{term}".

The word/phrase "{term}" MUST be written in every single post. This is mandatory.

What "{term}" means:
{context}

Categories: {cat_str} | Language origins: {lang_str}

For "hate" posts, target one of these groups:
{group_block}

Write exactly:
- {n_per_label} "hate" posts (targeting a specific racial/religious/migrant group)
- {n_per_label} "offensive" posts (rude but NOT targeting a protected group)
- {n_per_label} "normal" posts (neutral/benign usage)

Rules: every post MUST contain "{term}". Plain text only. Write in English Singlish only (English sentence structure; Singlish particles allowed). Keep each post 1-3 sentences.
Do NOT output full Malay/Chinese/Tamil sentences.

JSON array only, no other text:
[
  {{"text": "... {term} ...", "label": "hate", "target_group": "malay"}},
  {{"text": "... {term} ...", "label": "offensive", "target_group": "none"}},
  {{"text": "... {term} ...", "label": "normal", "target_group": "none"}}
]'''

def local_chat_generate(system_prompt, user_prompt, temperature=0.8, max_new_tokens=768):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    chat_text = LOCAL_TOKENIZER.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = LOCAL_TOKENIZER(chat_text, return_tensors='pt').to(LOCAL_MODEL.device)

    with torch.no_grad():
        outputs = LOCAL_MODEL.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            pad_token_id=LOCAL_TOKENIZER.eos_token_id,
        )

    gen_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return LOCAL_TOKENIZER.decode(gen_ids, skip_special_tokens=True).strip()

def parse_llm_output(raw_text):
    text = raw_text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)

    try:
        data = json.loads(text)
        if isinstance(data, dict):
            if 'text' in data and 'label' in data:
                data = [data]
            else:
                for v in data.values():
                    if isinstance(v, list):
                        data = v
                        break
                else:
                    return []
        if not isinstance(data, list):
            return []
    except json.JSONDecodeError:
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if not match:
            return []
        try:
            data = json.loads(match.group())
        except json.JSONDecodeError:
            return []

    valid = []
    for item in data:
        if isinstance(item, dict) and 'text' in item and 'label' in item:
            label = str(item['label']).strip().lower()
            if label in ('hate', 'offensive', 'normal'):
                if 'target_group' not in item:
                    item['target_group'] = 'none'
                item['label'] = label
                valid.append(item)
    return valid

def term_appears_in_text(term, text):
    text_lower = (text or '').lower()
    term_lower = term.lower().strip()

    if term_lower in text_lower:
        return True

    clean_term = term_lower.replace('…', '').replace('...', '')
    core_words = [w for w in clean_term.split() if len(w) > 1]
    if core_words:
        matches = sum(1 for w in core_words if w in text_lower)
        if matches == len(core_words):
            return True

    stop_words = {'buay', 'the', 'and', 'for', 'a', 'is', 'to', 'of', 'in'}
    words = [w for w in term_lower.split() if len(w) > 2 and w not in stop_words and '.' not in w and '…' not in w]
    if words:
        matches = sum(1 for w in words if w in text_lower)
        if matches >= max(1, len(words) * 0.6):
            return True

    if term.isupper() and len(term) <= 6 and term in (text or ''):
        return True

    return False

def preview_prompts(input_file, selected_terms, n_per_label=2, limit=3):
    entries = [normalize_entry(json.loads(line)) for line in Path(input_file).open(encoding='utf-8') if line.strip()]
    entries = [e for e in entries if e['term']]
    if selected_terms:
        emap = {e['term'].casefold(): e for e in entries}
        entries = [emap[t.casefold()] for t in selected_terms if t.casefold() in emap]

    for entry in entries[:limit]:
        groups = infer_target_groups(entry['languages'])
        prompt = build_prompt(entry['term'], entry['categories'], entry['context'], n_per_label, groups, entry['languages'])
        print('\n' + '='*80)
        print('TERM:', entry['term'])
        print('='*80)
        print(prompt)

def generate_dataset(input_file, output_file, selected_terms, n_per_label=2, limit=None):
    input_file = Path(input_file)
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    entries = [normalize_entry(json.loads(line)) for line in input_file.open(encoding='utf-8') if line.strip()]
    entries = [e for e in entries if e['term']]

    if selected_terms:
        emap = {e['term'].casefold(): e for e in entries}
        entries = [emap[t.casefold()] for t in selected_terms if t.casefold() in emap]

    if limit:
        entries = entries[:limit]

    total_written = 0
    with output_file.open('a', encoding='utf-8') as fh:
        for idx, entry in enumerate(entries, start=1):
            term = entry['term']
            if not entry['context']:
                print(f'[{idx}/{len(entries)}] {term} ... SKIP (no context)')
                continue

            groups = infer_target_groups(entry['languages'])
            prompt = build_prompt(term, entry['categories'], entry['context'], n_per_label, groups, entry['languages'])
            print(f'[{idx}/{len(entries)}] {term} ...', end=' ', flush=True)

            raw = local_chat_generate(SYSTEM_PROMPT, prompt)
            examples = parse_llm_output(raw)
            examples = [e for e in examples if term_appears_in_text(term, e.get('text', ''))]

            if not examples:
                print('FAILED')
                continue

            for ex in examples:
                fh.write(json.dumps({
                    'text': ex['text'],
                    'label': ex['label'],
                    'target_group': ex.get('target_group', 'none'),
                    'source_term': term,
                    'source_category': (entry['categories'][0] if entry['categories'] else ''),
                }, ensure_ascii=False) + '\n')
                total_written += 1

            print(f'OK ({len(examples)} rows)')
            time.sleep(0.2)

    print('Done. Wrote', total_written, 'rows to', output_file)

## Configure generation

In [6]:
# Minimal config
INPUT_FILE = DRIVE_PARSED_FILE  # /content/drive/MyDrive/dap-hs/singlish_parsed.jsonl
OUTPUT_FILE = DRIVE_OUTPUT_FILE # /content/drive/MyDrive/dap-hs/synthetic_training.jsonl

PER_TERM = 2
LIMIT = None

SELECTED_TERMS_RAW = '''
anak babi
apu neh neh
chao chee bai
diu lei lo mo
kan ni na bu
NBCB
pariah
pubor kia
pukimak
siao lang
tu lan
'''

SELECTED_TERMS = [t.strip() for t in re.split(r'[,\n;]', SELECTED_TERMS_RAW) if t.strip()]

print('Model:', HF_MODEL)
print('Input:', INPUT_FILE)
print('Output:', OUTPUT_FILE)
print('Terms:', len(SELECTED_TERMS), SELECTED_TERMS)

Model: Qwen/Qwen2.5-7B-Instruct
Input: /content/drive/MyDrive/dap-hs/singlish_parsed.jsonl
Output: /content/drive/MyDrive/dap-hs/synthetic_training.jsonl
Terms: 11 ['anak babi', 'apu neh neh', 'chao chee bai', 'diu lei lo mo', 'kan ni na bu', 'NBCB', 'pariah', 'pubor kia', 'pukimak', 'siao lang', 'tu lan']


## Dry run (preview prompt text)

In [7]:
preview_prompts(
    input_file=INPUT_FILE,
    selected_terms=SELECTED_TERMS,
    n_per_label=PER_TERM,
    limit=3,
)


TERM: anak babi
Write 6 Singapore social media posts that all use the Singlish term "anak babi".

The word/phrase "anak babi" MUST be written in every single post. This is mandatory.

What "anak babi" means:
Definitions:
1.   (pejorative) 'pig child'; 'child of a pig'; term of disdain, contempt, displeasure, anger, etc. especially used to scold a child

Court documents state that Madam Marliah was cleaning her home on April 24 last year when Lee became unhappy with the noise caused by her vacuuming. From her flat, she shouted the words "anak babi", which means "baby pig" in Malay, at Madam Marliah.

— 2020 Feb 6, Dominic Low. The Straits Times, "12 months' probation for woman who wounded neighbour's religious feelings".

Usage notes:
Particularly offensive to Muslims, due to the association of pigs with impurity in Islam.

Categories: insults | Language origins: mixed/unclear

For "hate" posts, target one of these groups:
- chinese: Chinese Singaporeans (majority race, includes dialec

## Generate dataset

In [8]:
generate_dataset(
    input_file=INPUT_FILE,
    output_file=OUTPUT_FILE,
    selected_terms=SELECTED_TERMS,
    n_per_label=PER_TERM,
    limit=LIMIT,
)

[1/11] anak babi ... OK (3 rows)
[2/11] apu neh neh ... OK (3 rows)
[3/11] chao chee bai ... OK (3 rows)
[4/11] diu lei lo mo ... OK (3 rows)
[5/11] kan ni na bu ... OK (3 rows)
[6/11] NBCB ... OK (6 rows)
[7/11] pariah ... OK (2 rows)
[8/11] pubor kia ... OK (6 rows)
[9/11] pukimak ... OK (6 rows)
[10/11] siao lang ... OK (3 rows)
[11/11] tu lan ... OK (6 rows)
Done. Wrote 44 rows to /content/drive/MyDrive/dap-hs/synthetic_training.jsonl


## Output check

In [9]:
import json
from collections import Counter, defaultdict
from pathlib import Path

out_file = Path(OUTPUT_FILE)
if not out_file.exists():
    raise FileNotFoundError(f'Output file not found: {out_file}')

rows = [json.loads(line) for line in out_file.open(encoding='utf-8') if line.strip()]
print('output file:', out_file)
print('rows:', len(rows))
print('unique terms:', len({r['source_term'] for r in rows}))
print('labels:', Counter(r['label'] for r in rows))

by_term = defaultdict(list)
for row in rows:
    by_term[row.get('source_term', 'UNKNOWN')].append(row)

available_terms = sorted(by_term.keys())
print('\nAvailable terms:')
for t in available_terms:
    print(f'- {t} ({len(by_term[t])} rows)')

# Set to a specific term (e.g. 'pukimak') or None to show all terms
TERM_TO_SHOW = None

def print_term_examples(term, examples):
    print('\n' + '=' * 100)
    print(f'TERM: {term} | rows: {len(examples)}')
    print('=' * 100)
    for idx, ex in enumerate(examples, start=1):
        print(f"[{idx}] label={ex.get('label')} | target_group={ex.get('target_group')}")
        print(ex.get('text', ''))
        print('-' * 100)

if TERM_TO_SHOW:
    if TERM_TO_SHOW not in by_term:
        raise ValueError(f"Term '{TERM_TO_SHOW}' not found. Choose from: {available_terms}")
    print_term_examples(TERM_TO_SHOW, by_term[TERM_TO_SHOW])
else:
    for term in available_terms:
        print_term_examples(term, by_term[term])

output file: /content/drive/MyDrive/dap-hs/synthetic_training.jsonl
rows: 110
unique terms: 11
labels: Counter({'offensive': 37, 'normal': 37, 'hate': 36})

Available terms:
- NBCB (12 rows)
- anak babi (9 rows)
- apu neh neh (9 rows)
- chao chee bai (9 rows)
- diu lei lo mo (9 rows)
- kan ni na bu (9 rows)
- pariah (8 rows)
- pubor kia (12 rows)
- pukimak (12 rows)
- siao lang (9 rows)
- tu lan (12 rows)

TERM: NBCB | rows: 12
[1] label=hate | target_group=chinese
You NBCB lah, why you always late for MRT? HDB flat too tight lor!
----------------------------------------------------------------------------------------------------
[2] label=offensive | target_group=none
NBCB you, you always forget to pay kopitiam bill! NS uniform too tight sia.
----------------------------------------------------------------------------------------------------
[3] label=normal | target_group=none
NBCB you, you look so tired after work. Need coffee, not NBCB!
---------------------------------------------